# 第 5 週實作：分類 — 邏輯迴歸、決策邊界與 ROC/PR 曲線
# Week 5 Lab: Classification — Logistic Regression, Decision Boundary & ROC/PR Curves

## 學習目標 Learning Objectives
1. 視覺化 Sigmoid 函數，理解其性質
2. 從零實作邏輯迴歸（使用梯度下降）
3. 視覺化決策邊界，觀察正則化參數 C 的影響
4. 繪製混淆矩陣熱力圖
5. 繪製 ROC 曲線與 PR 曲線，計算 AUC
6. 互動式閾值調整實驗
7. 不平衡資料處理示範

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline

# 設定中文字型（如果系統有的話）
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 設定繪圖風格
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# 設定隨機種子
np.random.seed(42)

print('所有套件匯入成功！ All packages imported successfully!')

---
## Part 1: Sigmoid 函數視覺化 Sigmoid Function Visualization

Sigmoid 函數 $\sigma(z) = \frac{1}{1 + e^{-z}}$ 是邏輯迴歸的核心，它將任意實數映射到 (0, 1) 區間。

In [ ]:
def sigmoid(z):
    """Sigmoid 函數"""
    return 1 / (1 + np.exp(-z))

# 繪製 Sigmoid 函數及其導數
z = np.linspace(-8, 8, 500)
sig = sigmoid(z)
sig_derivative = sig * (1 - sig)  # sigma'(z) = sigma(z) * (1 - sigma(z))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：Sigmoid 函數
ax1 = axes[0]
ax1.plot(z, sig, 'b-', linewidth=2.5, label=r'$\sigma(z) = \frac{1}{1+e^{-z}}$')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='y = 0.5')
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.2)
ax1.axhline(y=1, color='gray', linestyle='-', alpha=0.2)
# 標註關鍵點
ax1.plot(0, 0.5, 'ro', markersize=8, zorder=5)
ax1.annotate('(0, 0.5)', xy=(0, 0.5), xytext=(1.5, 0.35),
            fontsize=12, arrowprops=dict(arrowstyle='->', color='red'))
ax1.fill_between(z, 0, sig, alpha=0.1, color='blue')
ax1.set_xlabel('z (logit)', fontsize=12)
ax1.set_ylabel(r'$\sigma(z)$', fontsize=12)
ax1.set_title('Sigmoid Function', fontsize=14)
ax1.legend(fontsize=11)
ax1.set_ylim(-0.05, 1.05)
ax1.set_xlim(-8, 8)

# 右圖：Sigmoid 導數
ax2 = axes[1]
ax2.plot(z, sig_derivative, 'r-', linewidth=2.5,
         label=r"$\sigma'(z) = \sigma(z)(1-\sigma(z))$")
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.plot(0, 0.25, 'ro', markersize=8, zorder=5)
ax2.annotate('max = 0.25 at z=0', xy=(0, 0.25), xytext=(2, 0.22),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='red'))
ax2.fill_between(z, 0, sig_derivative, alpha=0.1, color='red')
ax2.set_xlabel('z (logit)', fontsize=12)
ax2.set_ylabel(r"$\sigma'(z)$", fontsize=12)
ax2.set_title('Sigmoid Derivative (Gradient)', fontsize=14)
ax2.legend(fontsize=11)
ax2.set_xlim(-8, 8)

plt.tight_layout()
plt.suptitle('Sigmoid 函數及其導數', fontsize=15, y=1.02)
plt.show()

print('觀察重點 Key Observations:')
print('1. Sigmoid 輸出範圍為 (0, 1)，可解釋為機率')
print('2. z=0 時輸出 0.5（決策邊界點）')
print('3. 導數在 z=0 時最大 (0.25)，|z| 大時趨近 0（梯度飽和/消失）')

In [ ]:
# 探索不同參數對 Sigmoid 形狀的影響
# sigma(w*z + b): w 控制陡峭程度，b 控制水平位移

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：不同 w 值（陡峭程度）
ax1 = axes[0]
for w in [0.5, 1, 2, 5]:
    ax1.plot(z, sigmoid(w * z), linewidth=2, label=f'w = {w}')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel('z', fontsize=12)
ax1.set_ylabel(r'$\sigma(wz)$', fontsize=12)
ax1.set_title('w 控制陡峭程度 (Steepness)', fontsize=13)
ax1.legend(fontsize=11)

# 右圖：不同 b 值（水平位移）
ax2 = axes[1]
for b in [-3, -1, 0, 1, 3]:
    ax2.plot(z, sigmoid(z + b), linewidth=2, label=f'b = {b}')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax2.set_xlabel('z', fontsize=12)
ax2.set_ylabel(r'$\sigma(z+b)$', fontsize=12)
ax2.set_title('b 控制水平位移 (Horizontal Shift)', fontsize=13)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print('w 越大 → 過渡越陡峭 → 模型越「果斷」（信心越高）')
print('b 控制曲線左右移動 → 對應決策邊界的偏移')

---
## Part 2: 從零實作邏輯迴歸 Logistic Regression from Scratch

我們將使用梯度下降法 (Gradient Descent) 從零實作二元邏輯迴歸，不依賴 scikit-learn。

**核心公式：**
- 預測：$\hat{p} = \sigma(Xw + b)$
- 損失：$\mathcal{L} = -\frac{1}{N}\sum[y\ln\hat{p} + (1-y)\ln(1-\hat{p})]$
- 梯度：$\nabla_w = \frac{1}{N}X^T(\hat{p} - y)$, $\nabla_b = \frac{1}{N}\sum(\hat{p} - y)$

In [ ]:
class LogisticRegressionScratch:
    """
    從零實作的邏輯迴歸 Logistic Regression from scratch
    使用梯度下降法最小化二元交叉熵損失
    """
    def __init__(self, learning_rate=0.1, n_iterations=1000, verbose=False):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        # 使用 clip 避免數值溢位
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def _compute_loss(self, y, y_pred):
        """計算二元交叉熵損失 Binary Cross-Entropy Loss"""
        epsilon = 1e-15  # 避免 log(0)
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        return loss

    def fit(self, X, y):
        """訓練模型 Train the model"""
        n_samples, n_features = X.shape

        # 初始化參數為零
        self.weights = np.zeros(n_features)
        self.bias = 0
        self.loss_history = []

        for i in range(self.n_iter):
            # 前向傳播 Forward pass
            z = X @ self.weights + self.bias
            y_pred = self._sigmoid(z)

            # 計算損失
            loss = self._compute_loss(y, y_pred)
            self.loss_history.append(loss)

            # 計算梯度 Compute gradients
            error = y_pred - y  # (N,)
            dw = (1 / n_samples) * (X.T @ error)   # (n_features,)
            db = (1 / n_samples) * np.sum(error)    # scalar

            # 更新參數 Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if self.verbose and (i + 1) % 100 == 0:
                print(f'Iteration {i+1}/{self.n_iter}, Loss: {loss:.6f}')

        return self

    def predict_proba(self, X):
        """預測機率 Predict probabilities"""
        z = X @ self.weights + self.bias
        return self._sigmoid(z)

    def predict(self, X, threshold=0.5):
        """預測類別 Predict classes"""
        return (self.predict_proba(X) >= threshold).astype(int)

print('LogisticRegressionScratch 類別定義完成！')

In [ ]:
# 生成二維分類資料 Generate 2D classification data
X, y = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    random_state=42, class_sep=1.5
)

# 標準化特徵
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 分割訓練/測試集
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

# 訓練自製模型
model_scratch = LogisticRegressionScratch(
    learning_rate=0.5, n_iterations=500, verbose=True
)
model_scratch.fit(X_train, y_train)

# 評估
y_pred_scratch = model_scratch.predict(X_test)
acc_scratch = accuracy_score(y_test, y_pred_scratch)
print(f'\n自製模型準確率 Scratch Model Accuracy: {acc_scratch:.4f}')
print(f'權重 Weights: {model_scratch.weights}')
print(f'偏置 Bias: {model_scratch.bias:.4f}')

In [ ]:
# 與 scikit-learn 比較 Compare with scikit-learn
model_sklearn = LogisticRegression(random_state=42, max_iter=500)
model_sklearn.fit(X_train, y_train)

y_pred_sklearn = model_sklearn.predict(X_test)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print('=== 模型比較 Model Comparison ===')
print(f'自製模型 Scratch  → Accuracy: {acc_scratch:.4f}, '
      f'Weights: [{model_scratch.weights[0]:.4f}, {model_scratch.weights[1]:.4f}], '
      f'Bias: {model_scratch.bias:.4f}')
print(f'sklearn 模型      → Accuracy: {acc_sklearn:.4f}, '
      f'Weights: [{model_sklearn.coef_[0][0]:.4f}, {model_sklearn.coef_[0][1]:.4f}], '
      f'Bias: {model_sklearn.intercept_[0]:.4f}')

In [ ]:
# 繪製訓練損失曲線 Plot training loss curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(model_scratch.loss_history, 'b-', linewidth=1.5)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('BCE Loss', fontsize=12)
ax.set_title('Training Loss Curve (Binary Cross-Entropy)', fontsize=14)
ax.axhline(y=model_scratch.loss_history[-1], color='r',
           linestyle='--', alpha=0.5, label=f'Final Loss: {model_scratch.loss_history[-1]:.4f}')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 3: 決策邊界視覺化 Decision Boundary Visualization

決策邊界 (Decision Boundary) 是將不同類別分開的曲線/直線。
對於邏輯迴歸，當閾值為 0.5 時，決策邊界是 $w^Tx + b = 0$。

In [ ]:
def plot_decision_boundary(model, X, y, ax=None, title='Decision Boundary',
                           use_sklearn=True, resolution=200):
    """
    繪製決策邊界 Plot decision boundary for 2D data
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # 建立網格 Create mesh grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    # 預測每個網格點的機率
    if use_sklearn:
        Z = model.predict_proba(grid)[:, 1]
    else:
        Z = model.predict_proba(grid)
    Z = Z.reshape(xx.shape)

    # 繪製機率等高線 Plot probability contours
    contour = ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 21),
                          cmap='RdYlBu_r', alpha=0.7)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black',
               linewidths=2, linestyles='--')

    # 繪製資料點 Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu_r',
                        edgecolors='black', s=40, alpha=0.8)

    ax.set_xlabel('Feature 1 ($x_1$)', fontsize=11)
    ax.set_ylabel('Feature 2 ($x_2$)', fontsize=11)
    ax.set_title(title, fontsize=13)

    return contour


# 繪製自製模型與 sklearn 模型的決策邊界比較
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

contour1 = plot_decision_boundary(
    model_scratch, X_scaled, y, axes[0],
    title='Scratch Model Decision Boundary', use_sklearn=False
)
plt.colorbar(contour1, ax=axes[0], label='P(y=1)')

contour2 = plot_decision_boundary(
    model_sklearn, X_scaled, y, axes[1],
    title='scikit-learn Model Decision Boundary', use_sklearn=True
)
plt.colorbar(contour2, ax=axes[1], label='P(y=1)')

plt.tight_layout()
plt.show()

print('黑色虛線 = 決策邊界 (P=0.5)')
print('顏色深淺 = 預測機率')
print('兩個模型的決策邊界應非常相似！')

---
## Part 4: 調整 C 值觀察決策邊界變化 Effect of Regularization Parameter C

在 scikit-learn 的 `LogisticRegression` 中，`C` 是正則化強度的**倒數** ($C = 1/\lambda$)。
- C 越小 → 正則化越強 → 決策邊界越簡單
- C 越大 → 正則化越弱 → 決策邊界可能越複雜（尤其搭配多項式特徵）

In [ ]:
# 使用 make_moons 產生非線性可分的資料
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)
scaler_moon = StandardScaler()
X_moon_scaled = scaler_moon.fit_transform(X_moon)

# 不同 C 值的比較（搭配多項式特徵以展示差異）
C_values = [0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))

for idx, C in enumerate(C_values):
    # 建立 Pipeline：多項式特徵 + 邏輯迴歸
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(C=C, max_iter=5000, random_state=42))
    ])
    pipe.fit(X_moon_scaled, y_moon)
    acc = pipe.score(X_moon_scaled, y_moon)

    ax = axes[idx]
    # 繪製決策邊界
    x_min, x_max = X_moon_scaled[:, 0].min() - 0.5, X_moon_scaled[:, 0].max() + 0.5
    y_min, y_max = X_moon_scaled[:, 1].min() - 0.5, X_moon_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    Z = pipe.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 21), cmap='RdYlBu_r', alpha=0.7)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X_moon_scaled[:, 0], X_moon_scaled[:, 1],
              c=y_moon, cmap='RdYlBu_r', edgecolors='black', s=20)
    ax.set_title(f'C = {C}\nAcc = {acc:.3f}', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('C 值對決策邊界的影響 (3 次多項式特徵)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('C 越小 → 強正則化 → 邊界越平滑（可能欠擬合）')
print('C 越大 → 弱正則化 → 邊界越複雜（可能過擬合）')

---
## Part 5: 混淆矩陣熱力圖 Confusion Matrix Heatmap

混淆矩陣 (Confusion Matrix) 是分類評估的基石，直觀顯示 TP、TN、FP、FN 的數量。

In [ ]:
# 使用較大的資料集進行混淆矩陣分析
X_clf, y_clf = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42, flip_y=0.1
)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

# 訓練模型
scaler_clf = StandardScaler()
X_train_clf_s = scaler_clf.fit_transform(X_train_clf)
X_test_clf_s = scaler_clf.transform(X_test_clf)

lr_clf = LogisticRegression(random_state=42, max_iter=1000)
lr_clf.fit(X_train_clf_s, y_train_clf)
y_pred_clf = lr_clf.predict(X_test_clf_s)

# 計算混淆矩陣
cm = confusion_matrix(y_test_clf, y_pred_clf)

# 繪製熱力圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：原始數量
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred Neg (0)', 'Pred Pos (1)'],
            yticklabels=['Actual Neg (0)', 'Actual Pos (1)'],
            annot_kws={'size': 16})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)

# 右圖：百分比
cm_pct = cm.astype('float') / cm.sum() * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['Pred Neg (0)', 'Pred Pos (1)'],
            yticklabels=['Actual Neg (0)', 'Actual Pos (1)'],
            annot_kws={'size': 16})
axes[1].set_title('Confusion Matrix (Percentages %)', fontsize=13)
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.show()

# 列印分類報告
print('\n=== Classification Report ===')
print(classification_report(y_test_clf, y_pred_clf, target_names=['Negative', 'Positive']))

tn, fp, fn, tp = cm.ravel()
print(f'TP (True Positive)  = {tp}')
print(f'TN (True Negative)  = {tn}')
print(f'FP (False Positive) = {fp}')
print(f'FN (False Negative) = {fn}')

---
## Part 6: ROC 曲線繪製與 AUC 計算 ROC Curve and AUC

ROC 曲線展示在不同閾值下 TPR (True Positive Rate) 與 FPR (False Positive Rate) 的權衡。
AUC (Area Under the Curve) 是 ROC 曲線下的面積，衡量模型的整體區分能力。

In [ ]:
# 取得預測機率
y_proba = lr_clf.predict_proba(X_test_clf_s)[:, 1]

# 計算 ROC 曲線
fpr, tpr, thresholds_roc = roc_curve(y_test_clf, y_proba)
roc_auc = auc(fpr, tpr)

# 找到 Youden's J 最佳閾值
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold_roc = thresholds_roc[best_idx]

# 繪製 ROC 曲線
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(fpr, tpr, 'b-', linewidth=2.5,
        label=f'Logistic Regression (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5,
        label='Random Classifier (AUC = 0.5)')

# 標註最佳閾值點
ax.plot(fpr[best_idx], tpr[best_idx], 'ro', markersize=12, zorder=5,
        label=f'Best Threshold = {best_threshold_roc:.3f}\n'
              f'(TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f})')

# 填充 AUC 區域
ax.fill_between(fpr, tpr, alpha=0.15, color='blue')

ax.set_xlabel('False Positive Rate (FPR)', fontsize=13)
ax.set_ylabel('True Positive Rate (TPR / Recall)', fontsize=13)
ax.set_title('ROC Curve (Receiver Operating Characteristic)', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'AUC = {roc_auc:.4f}')
print(f'最佳閾值 (Youden\'s J) = {best_threshold_roc:.4f}')
print(f'  → TPR = {tpr[best_idx]:.4f}, FPR = {fpr[best_idx]:.4f}')

---
## Part 7: PR 曲線繪製 Precision-Recall Curve

PR 曲線在不平衡資料中比 ROC 曲線更具參考價值。
AP (Average Precision) 是 PR 曲線下的面積。

In [ ]:
# 計算 PR 曲線
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_test_clf, y_proba)
ap = average_precision_score(y_test_clf, y_proba)

# 計算每個閾值下的 F1-Score，找最佳閾值
f1_scores = 2 * (precision_curve[:-1] * recall_curve[:-1]) / \
            (precision_curve[:-1] + recall_curve[:-1] + 1e-10)
best_f1_idx = np.argmax(f1_scores)
best_threshold_pr = thresholds_pr[best_f1_idx]

# 繪製 PR 曲線
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：PR 曲線
ax1 = axes[0]
ax1.plot(recall_curve, precision_curve, 'g-', linewidth=2.5,
         label=f'Logistic Regression (AP = {ap:.4f})')
# 基準線（隨機分類器的 Precision = 正類比例）
baseline = np.mean(y_test_clf)
ax1.axhline(y=baseline, color='gray', linestyle='--', alpha=0.5,
            label=f'Random Baseline (P = {baseline:.2f})')
ax1.plot(recall_curve[best_f1_idx], precision_curve[best_f1_idx],
         'ro', markersize=12, zorder=5,
         label=f'Best F1 Threshold = {best_threshold_pr:.3f}\n'
               f'(P={precision_curve[best_f1_idx]:.3f}, R={recall_curve[best_f1_idx]:.3f})')
ax1.fill_between(recall_curve, precision_curve, alpha=0.15, color='green')
ax1.set_xlabel('Recall', fontsize=13)
ax1.set_ylabel('Precision', fontsize=13)
ax1.set_title('Precision-Recall Curve', fontsize=14)
ax1.legend(loc='lower left', fontsize=10)
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([0, 1.05])
ax1.grid(True, alpha=0.3)

# 右圖：F1-Score vs Threshold
ax2 = axes[1]
ax2.plot(thresholds_pr, f1_scores, 'purple', linewidth=2,
         label='F1-Score')
ax2.plot(thresholds_pr, precision_curve[:-1], 'b--', linewidth=1.5,
         label='Precision', alpha=0.7)
ax2.plot(thresholds_pr, recall_curve[:-1], 'r--', linewidth=1.5,
         label='Recall', alpha=0.7)
ax2.axvline(x=best_threshold_pr, color='green', linestyle=':',
            label=f'Best Threshold = {best_threshold_pr:.3f}')
ax2.set_xlabel('Threshold', fontsize=13)
ax2.set_ylabel('Score', fontsize=13)
ax2.set_title('Precision / Recall / F1 vs. Threshold', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Average Precision (AP) = {ap:.4f}')
print(f'Best F1 Threshold = {best_threshold_pr:.4f}')
print(f'  → Precision = {precision_curve[best_f1_idx]:.4f}')
print(f'  → Recall = {recall_curve[best_f1_idx]:.4f}')
print(f'  → F1 = {f1_scores[best_f1_idx]:.4f}')

---
## Part 8: 閾值調整互動實驗 Threshold Adjustment Experiment

改變閾值 (Threshold) 會如何影響混淆矩陣和各項指標？
讓我們系統地觀察閾值對分類結果的影響。

In [ ]:
# 閾值調整的視覺化實驗
thresholds_experiment = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

fig, axes = plt.subplots(2, len(thresholds_experiment), figsize=(28, 8))

results = []

for idx, thresh in enumerate(thresholds_experiment):
    y_pred_thresh = (y_proba >= thresh).astype(int)
    cm_thresh = confusion_matrix(y_test_clf, y_pred_thresh)

    acc = accuracy_score(y_test_clf, y_pred_thresh)
    prec = precision_score(y_test_clf, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test_clf, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test_clf, y_pred_thresh, zero_division=0)

    results.append({
        'Threshold': thresh, 'Accuracy': acc,
        'Precision': prec, 'Recall': rec, 'F1': f1
    })

    # 上排：混淆矩陣
    sns.heatmap(cm_thresh, annot=True, fmt='d', cmap='Blues',
                ax=axes[0, idx], cbar=False,
                xticklabels=['N', 'P'], yticklabels=['N', 'P'],
                annot_kws={'size': 14})
    axes[0, idx].set_title(f'Threshold = {thresh}', fontsize=11, fontweight='bold')
    if idx == 0:
        axes[0, idx].set_ylabel('True', fontsize=10)

    # 下排：指標長條圖
    metrics = [acc, prec, rec, f1]
    labels = ['Acc', 'Prec', 'Rec', 'F1']
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
    bars = axes[1, idx].bar(labels, metrics, color=colors, alpha=0.8)
    axes[1, idx].set_ylim(0, 1.05)
    for bar, val in zip(bars, metrics):
        axes[1, idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                         f'{val:.2f}', ha='center', fontsize=9)
    axes[1, idx].set_title(f'F1 = {f1:.3f}', fontsize=10)

plt.suptitle('Threshold Effect on Confusion Matrix & Metrics', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# 顯示結果表格
df_results = pd.DataFrame(results)
print('\n=== Threshold Comparison Table ===')
print(df_results.to_string(index=False, float_format='%.4f'))
print(f'\nBest F1 at threshold = {df_results.loc[df_results["F1"].idxmax(), "Threshold"]}')

In [ ]:
# 預測機率分布圖（按真實類別分開）
fig, ax = plt.subplots(figsize=(10, 5))

# 分開正負類別的預測機率
proba_neg = y_proba[y_test_clf == 0]
proba_pos = y_proba[y_test_clf == 1]

ax.hist(proba_neg, bins=30, alpha=0.6, color='blue', label='Actual Negative (y=0)',
        edgecolor='black', density=True)
ax.hist(proba_pos, bins=30, alpha=0.6, color='red', label='Actual Positive (y=1)',
        edgecolor='black', density=True)

# 繪製不同閾值線
for thresh, ls in [(0.3, ':'), (0.5, '--'), (0.7, '-.')]:
    ax.axvline(x=thresh, color='green', linestyle=ls,
              linewidth=2, label=f'Threshold = {thresh}')

ax.set_xlabel('Predicted Probability P(y=1)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Predicted Probability Distribution by True Class', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('理想情況：兩個分布完全分離，任何閾值都能完美分類')
print('實際情況：兩個分布有重疊，閾值選擇會在 FP 和 FN 之間取捨')

---
## Part 9: 不平衡資料處理示範 Imbalanced Data Handling

在真實世界中，許多分類問題的資料是不平衡的。
讓我們觀察不平衡資料的影響，以及各種處理策略的效果。

In [ ]:
# 建立不平衡資料集（正類只佔 5%）
X_imb, y_imb = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2,
    weights=[0.95, 0.05],  # 95% 負類, 5% 正類
    random_state=42, flip_y=0.02
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

scaler_imb = StandardScaler()
X_train_imb_s = scaler_imb.fit_transform(X_train_imb)
X_test_imb_s = scaler_imb.transform(X_test_imb)

print(f'訓練集 Training set:')
print(f'  Negative (y=0): {np.sum(y_train_imb == 0)} ({np.mean(y_train_imb == 0)*100:.1f}%)')
print(f'  Positive (y=1): {np.sum(y_train_imb == 1)} ({np.mean(y_train_imb == 1)*100:.1f}%)')
print(f'  Imbalance Ratio: 1:{np.sum(y_train_imb == 0) // max(np.sum(y_train_imb == 1), 1)}')

In [ ]:
# 比較不同策略
from collections import Counter

strategies = {}

# 策略 1: 不做處理（基線）
lr_base = LogisticRegression(random_state=42, max_iter=1000)
lr_base.fit(X_train_imb_s, y_train_imb)
strategies['Baseline (No handling)'] = lr_base

# 策略 2: Class Weight = balanced
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_balanced.fit(X_train_imb_s, y_train_imb)
strategies['class_weight=balanced'] = lr_balanced

# 策略 3: 手動 SMOTE（簡化版，不需 imblearn）
def simple_smote(X, y, target_ratio=1.0, k=5, random_state=42):
    """
    簡化版 SMOTE 實作 Simplified SMOTE implementation
    """
    rng = np.random.RandomState(random_state)
    X_minority = X[y == 1]
    X_majority = X[y == 0]
    n_minority = len(X_minority)
    n_majority = len(X_majority)
    n_synthetic = int(n_majority * target_ratio) - n_minority

    if n_synthetic <= 0:
        return X, y

    # 對每個少數類樣本找 k 近鄰
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=min(k + 1, n_minority))
    nn.fit(X_minority)

    synthetic_samples = []
    for _ in range(n_synthetic):
        idx = rng.randint(0, n_minority)
        distances, neighbors = nn.kneighbors([X_minority[idx]])
        neighbor_idx = rng.choice(neighbors[0][1:])  # 排除自己
        diff = X_minority[neighbor_idx] - X_minority[idx]
        lam = rng.random()
        new_sample = X_minority[idx] + lam * diff
        synthetic_samples.append(new_sample)

    X_synthetic = np.array(synthetic_samples)
    X_resampled = np.vstack([X, X_synthetic])
    y_resampled = np.hstack([y, np.ones(n_synthetic)])

    return X_resampled, y_resampled

# 應用 SMOTE
X_smote, y_smote = simple_smote(X_train_imb_s, y_train_imb, target_ratio=0.5)
print(f'SMOTE 後類別分布: {Counter(y_smote.astype(int))}')

lr_smote = LogisticRegression(random_state=42, max_iter=1000)
lr_smote.fit(X_smote, y_smote)
strategies['SMOTE'] = lr_smote

# 策略 4: 隨機欠採樣 Random Undersampling
def random_undersample(X, y, ratio=1.0, random_state=42):
    rng = np.random.RandomState(random_state)
    X_minority = X[y == 1]
    X_majority = X[y == 0]
    n_keep = int(len(X_minority) * ratio / 1.0)
    n_keep = min(n_keep, len(X_majority))
    idx = rng.choice(len(X_majority), n_keep, replace=False)
    X_resampled = np.vstack([X_majority[idx], X_minority])
    y_resampled = np.hstack([np.zeros(n_keep), np.ones(len(X_minority))])
    return X_resampled, y_resampled

X_under, y_under = random_undersample(X_train_imb_s, y_train_imb, ratio=2.0)
print(f'Undersampling 後類別分布: {Counter(y_under.astype(int))}')

lr_under = LogisticRegression(random_state=42, max_iter=1000)
lr_under.fit(X_under, y_under)
strategies['Random Undersampling'] = lr_under

In [ ]:
# 比較各策略的結果
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

comparison_results = []

for idx, (name, model) in enumerate(strategies.items()):
    ax_row = idx // 2
    ax_col = idx % 2
    ax = axes[ax_row, ax_col]

    y_pred_s = model.predict(X_test_imb_s)
    y_proba_s = model.predict_proba(X_test_imb_s)[:, 1]

    cm_s = confusion_matrix(y_test_imb, y_pred_s)
    sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Neg', 'Pred Pos'],
                yticklabels=['True Neg', 'True Pos'],
                annot_kws={'size': 14})

    acc_s = accuracy_score(y_test_imb, y_pred_s)
    prec_s = precision_score(y_test_imb, y_pred_s, zero_division=0)
    rec_s = recall_score(y_test_imb, y_pred_s, zero_division=0)
    f1_s = f1_score(y_test_imb, y_pred_s, zero_division=0)
    ap_s = average_precision_score(y_test_imb, y_proba_s)
    fpr_s, tpr_s, _ = roc_curve(y_test_imb, y_proba_s)
    auc_s = auc(fpr_s, tpr_s)

    ax.set_title(f'{name}\nAcc={acc_s:.3f} | P={prec_s:.3f} | R={rec_s:.3f} | '
                f'F1={f1_s:.3f} | AUC={auc_s:.3f}', fontsize=11)

    comparison_results.append({
        'Strategy': name, 'Accuracy': acc_s, 'Precision': prec_s,
        'Recall': rec_s, 'F1': f1_s, 'AUC': auc_s, 'AP': ap_s
    })

plt.suptitle('Imbalanced Data: Strategy Comparison', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# 顯示比較表格
df_comparison = pd.DataFrame(comparison_results)
print('\n=== Strategy Comparison ===')
print(df_comparison.to_string(index=False, float_format='%.4f'))
print('\n注意：Baseline 的 Accuracy 很高但 Recall 很低 — 它幾乎把所有樣本都預測為負類！')
print('class_weight, SMOTE, Undersampling 都能提升少數類的 Recall。')

In [ ]:
# ROC 和 PR 曲線比較各策略
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, (name, model) in enumerate(strategies.items()):
    y_proba_s = model.predict_proba(X_test_imb_s)[:, 1]

    # ROC
    fpr_s, tpr_s, _ = roc_curve(y_test_imb, y_proba_s)
    auc_s = auc(fpr_s, tpr_s)
    axes[0].plot(fpr_s, tpr_s, color=colors[idx], linewidth=2,
                label=f'{name} (AUC={auc_s:.3f})')

    # PR
    prec_s, rec_s, _ = precision_recall_curve(y_test_imb, y_proba_s)
    ap_s = average_precision_score(y_test_imb, y_proba_s)
    axes[1].plot(rec_s, prec_s, color=colors[idx], linewidth=2,
                label=f'{name} (AP={ap_s:.3f})')

# ROC 圖設定
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('FPR', fontsize=12)
axes[0].set_ylabel('TPR', fontsize=12)
axes[0].set_title('ROC Curve Comparison', fontsize=14)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# PR 圖設定
baseline_pr = np.mean(y_test_imb)
axes[1].axhline(y=baseline_pr, color='gray', linestyle='--', alpha=0.3,
               label=f'Baseline (P={baseline_pr:.2f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('PR Curve Comparison (More Informative for Imbalanced Data!)', fontsize=14)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('觀察重點：')
print('1. ROC 曲線中各策略的 AUC 差異較小 — ROC 可能過於樂觀')
print('2. PR 曲線更能區分不同策略的效果差異')
print('3. 在不平衡資料中，PR 曲線是更可靠的評估工具')

---
## Part 10: 練習題 Exercises

完成以下練習來鞏固本週所學。

### 練習 1: Sigmoid 函數的性質驗證

驗證 Sigmoid 函數的對稱性質：$\sigma(-z) = 1 - \sigma(z)$

提示：使用 numpy 計算幾個 z 值的 sigma(z) 和 sigma(-z)，確認它們的和為 1。

In [ ]:
# 練習 1: 請在此撰寫你的程式碼
# TODO: 驗證 sigma(-z) = 1 - sigma(z)

# z_values = np.array([-3, -1, 0, 1, 3])
# ...

### 練習 2: 手動計算 BCE Loss

給定以下預測和真實標籤，手動計算 BCE Loss：
- y_true = [1, 0, 1, 1, 0]
- y_pred_proba = [0.9, 0.2, 0.7, 0.4, 0.1]

提示：$\mathcal{L} = -\frac{1}{N}\sum[y\ln\hat{p} + (1-y)\ln(1-\hat{p})]$

In [ ]:
# 練習 2: 請在此撰寫你的程式碼
# TODO: 計算 BCE Loss

# y_true = np.array([1, 0, 1, 1, 0])
# y_pred_proba = np.array([0.9, 0.2, 0.7, 0.4, 0.1])
# ...

### 練習 3: 使用 Breast Cancer 資料集

使用 `sklearn.datasets.load_breast_cancer` 資料集：
1. 訓練一個邏輯迴歸模型
2. 繪製混淆矩陣
3. 繪製 ROC 和 PR 曲線
4. 找出使 F1-Score 最大化的最佳閾值

提示：這是一個真實的醫療資料集，思考在這個場景下，Precision 和 Recall 哪個更重要？

In [ ]:
# 練習 3: 請在此撰寫你的程式碼
# from sklearn.datasets import load_breast_cancer
# TODO: 完成上述四個步驟

# data = load_breast_cancer()
# X_bc, y_bc = data.data, data.target
# ...

### 練習 4: 不同 C 值的交叉驗證

對 Breast Cancer 資料集，使用 5-fold 交叉驗證 (Cross-Validation) 找出最佳的 C 值。
嘗試的 C 值：[0.001, 0.01, 0.1, 1, 10, 100]

提示：使用 `sklearn.model_selection.cross_val_score`

In [ ]:
# 練習 4: 請在此撰寫你的程式碼
# from sklearn.model_selection import cross_val_score
# TODO: 交叉驗證找最佳 C

# C_candidates = [0.001, 0.01, 0.1, 1, 10, 100]
# ...

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **Sigmoid 函數視覺化**：理解其形狀、性質與參數影響
2. **從零實作邏輯迴歸**：手寫梯度下降，並與 scikit-learn 比較
3. **決策邊界視覺化**：觀察線性邊界與機率等高線
4. **C 值實驗**：搭配多項式特徵觀察正則化對邊界的影響
5. **混淆矩陣熱力圖**：直觀理解 TP/TN/FP/FN
6. **ROC 曲線 + AUC**：衡量模型整體區分能力
7. **PR 曲線 + AP**：不平衡資料的可靠評估
8. **閾值調整實驗**：觀察不同閾值對指標的影響
9. **不平衡資料處理**：比較 Baseline、Class Weight、SMOTE、Undersampling

### 關鍵收穫 Key Takeaways
- 邏輯迴歸的本質是「線性模型 + Sigmoid = 機率輸出」
- 閾值選擇不是模型的一部分，而是**部署決策**
- 不平衡資料中，Accuracy 會誤導你，要看 **F1、PR-AUC** 等指標
- 沒有萬能的評估指標，選擇取決於**業務場景**和**錯誤代價**